# Monthly Revenue Trend

In [0]:
SELECT 
    year,
    month_name,
    sum(total_revenue) as revenue_usd
FROM retailer.gold.data_cube
WHERE year = 2019 GROUP BY year, month_name;

# Peak Month Analysis (Top 3 + %)

In [0]:
WITH monthly AS (
    SELECT 
        month,
        SUM(total_revenue) AS revenue
    FROM retailer.gold.data_cube
    WHERE year = 2019
    GROUP BY month
),
total AS (
    SELECT SUM(revenue) AS total_revenue FROM monthly
)
SELECT 
    m.month,
    m.revenue,
    ROUND((m.revenue / t.total_revenue) * 100, 2) AS pct_of_total
FROM monthly m, total t
ORDER BY m.revenue DESC
LIMIT 3;

# Holiday Drivers

In [0]:
WITH monthly AS (
    SELECT 
        month,
        SUM(total_revenue) AS revenue
    FROM retailer.gold.data_cube
    WHERE year = 2019 AND month IN (4,5,6)
    GROUP BY month
),
top_months AS (
    SELECT month
    FROM monthly
    ORDER BY revenue DESC
    LIMIT 2
)
SELECT 
    category,
    SUM(total_revenue) AS revenue
FROM retailer.gold.data_cube
WHERE month IN (SELECT month FROM top_months)
GROUP BY category
ORDER BY revenue DESC
LIMIT 3;

# Delivery Performance

In [0]:
SELECT 
  ROUND(SUM(avg_delivery_days * order_count) / SUM(order_count), 2) AS average_days,
  SUM(order_count) AS total_orders
FROM retailer.gold.data_cube
WHERE avg_delivery_days IS NOT NULL
     

# Country Delivery Issues (Slowest 5)

In [0]:
SELECT 
  customer_country AS country,
  ROUND(SUM(avg_delivery_days * order_count) / SUM(order_count), 2) AS avg_days,
  SUM(order_count) AS order_count
FROM retailer.gold.data_cube
WHERE avg_delivery_days IS NOT NULL
GROUP BY customer_country
ORDER BY avg_days DESC

# Channel Performance

In [0]:
WITH channel_aov AS (
  SELECT 
    continent,
    CASE WHEN store_country IS NULL THEN 'Online' ELSE 'In-Store' END AS channel,
    ROUND(SUM(total_revenue) / SUM(order_count), 2) AS aov,
    SUM(order_count) AS orders
  FROM retailer.gold.data_cube
  GROUP BY continent, CASE WHEN store_country IS NULL THEN 'Online' ELSE 'In-Store' END
)
SELECT 
  continent,
  COALESCE(MAX(CASE WHEN channel = 'Online' THEN aov END), 0) AS aov_online,
  COALESCE(MAX(CASE WHEN channel = 'Online' THEN orders END), 0) AS online_orders,
  MAX(CASE WHEN channel = 'In-Store' THEN aov END) AS aov_store,
  MAX(CASE WHEN channel = 'In-Store' THEN orders END) AS store_orders
FROM channel_aov
GROUP BY continent
ORDER BY continent

# Volume Leaders

In [0]:
SELECT 
    category,
    SUM(total_quantity) AS total_units
FROM retailer.gold.data_cube
GROUP BY category
ORDER BY total_units DESC
LIMIT 5;

# Revenue Leaders

In [0]:
SELECT 
    category,
    SUM(total_revenue) AS total_revenue
FROM retailer.gold.data_cube
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 5;

# Customer Profile

In [0]:
SELECT 
    c.continent,
    c.gender,
    COUNT(DISTINCT c.customer_key) AS customer_count,
    SUM(f.revenue_usd) AS total_spending
FROM retailer.gold.fact_sales f
JOIN retailer.gold.dim_customers c
    ON f.customer_key = c.customer_key
GROUP BY c.continent, c.gender
ORDER BY total_spending DESC;

# Customer Loyalty

In [0]:
WITH customer_orders AS (
    SELECT 
        customer_key,
        COUNT(DISTINCT order_number) AS order_count
    FROM retailer.gold.fact_sales
    GROUP BY customer_key
),
repeat_customers AS (
    SELECT customer_key
    FROM customer_orders
    WHERE order_count >= 2
)
SELECT 
    c.continent,
    COUNT(DISTINCT r.customer_key) * 100.0 / 
    COUNT(DISTINCT c.customer_key) AS repeat_rate_pct
FROM retailer.gold.dim_customers c
LEFT JOIN repeat_customers r
    ON c.customer_key = r.customer_key
GROUP BY c.continent;